# 推理型 LLM 的低比特量化：为什么同一个量化模型有两条完全不同的掉点曲线

对应文章：《大模型量化算法（22）：Reasoning LLM 低比特量化》
https://lrypcy.github.io/2026/09/19/llm-quant-22-reasoning-llm-lowbit/

| 实验 | 要回答的问题 |
|---|---|
| A | 长链误差放大：单步错误率在 PPL 上看不出来，为什么长链上会崩？ |
| B | 两个域的形状差异 = 量化代价差异：推理域的重尾激活 |
| C | 解法一：混合域校准。为什么是 80/20，而不是 100/0？ |
| D | 解法二：渐进量化。FP16→INT4→INT2 vs 直接 INT2 |
| E | 解法三：教师引导的奖励修正（配合 20 篇 §4.3） |
| F | 诊断：分域评测 + 长度扫描，而不是只看平均分 |

纯 numpy 合成探针，SEED=0。

In [1]:
import os, json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 0
MODE = "smoke"          # "smoke" | "full"
CFG = {
    "smoke": dict(chain_lens=(3, 16, 32, 64, 128, 512), bits=(2, 3, 4, 6, 8),
                  d=128, n_cal=1500, rho_grid=(0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.5, 0.8, 1.0),
                  prog_steps=300, n_trial=2000),
    "full":  dict(chain_lens=(3, 8, 16, 32, 64, 128, 256, 512, 1024), bits=(2, 3, 4, 6, 8),
                  d=256, n_cal=4000, rho_grid=(0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0),
                  prog_steps=800, n_trial=8000),
}[MODE]
HERE = os.getcwd(); RES = os.path.join(HERE, "results"); os.makedirs(RES, exist_ok=True)
_LINES = []
def log(m=""):
    print(m); _LINES.append(str(m))
def savefig(fig, name):
    p = os.path.join(RES, name); fig.savefig(p, dpi=130, bbox_inches="tight"); plt.close(fig)
    log(f"[save] {p}"); return p
def sym_quant(X, b, group=None):
    qmax = 2 ** (b - 1) - 1
    if group is None:
        s = np.abs(X).max() / qmax
        return np.round(X / s) * s
    Xf = X.reshape(-1, group)
    s = np.abs(Xf).max(axis=1, keepdims=True) / qmax
    return (np.round(Xf / s) * s).reshape(X.shape)
def rel_err(X, Xq):
    return float(np.linalg.norm(X - Xq) / np.linalg.norm(X))
log(f"MODE={MODE} CFG={CFG}")

MODE=smoke CFG={'chain_lens': (3, 16, 32, 64, 128, 512), 'bits': (2, 3, 4, 6, 8), 'd': 128, 'n_cal': 1500, 'rho_grid': (0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.5, 0.8, 1.0), 'prog_steps': 300, 'n_trial': 2000}


## A · 长链误差放大

设单步错误率 $\varepsilon$。链长 $L$，全链正确率 $=(1-\varepsilon)^L$。
取 $\varepsilon=0.005$（**PPL 上完全看不出来的量级**），看 $L$ 增大时全链正确率怎么塌。

更真实一点：每步错误率随上下文增长（误差累积，$\varepsilon_t=\varepsilon(1+\log(1+t))$），
掉得更快。这就是「量化在短回答任务上看起来没事、在长推理链上崩掉」的机理。

In [2]:
eps = 0.005
log("=== A 长链误差放大 ===")
rows_A = []
for L in CFG["chain_lens"]:
    const = (1 - eps) ** L
    per_step = eps * (1 + np.log(1 + np.arange(L)))       # 越往后越容易错
    accum = float(np.prod(1 - per_step))
    rows_A.append(dict(L=L, const=float(const), accum=accum))
    log(f"  链长 L={L:>5}   恒定错误率全链正确率={const:.4f}   累积模型={accum:.4f}")
log("-" * 78)
log(f"  读数：eps={eps} 时 PPL/单步指标毫无异常，但 L=512 全链正确率已跌到 "
    f"{(1-eps)**512:.3f}（累积模型更低）。")
log("        => 推理模型必须用『全链正确率 / Pass@1 on long CoT』评测，短回答指标会骗人。")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot([r["L"] for r in rows_A], [r["const"] for r in rows_A], marker="o", label="constant per-step eps")
ax[0].plot([r["L"] for r in rows_A], [r["accum"] for r in rows_A], marker="s", label="accumulating eps")
ax[0].axhline(0.5, color="grey", ls="--", lw=0.8)
ax[0].set_xscale("log", base=2); ax[0].set_xlabel("chain length L")
ax[0].set_ylabel("P(all steps correct)")
ax[0].set_title(f"[A] eps={eps} looks harmless, L=512 is not"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[1].plot([r["L"] for r in rows_A], [r["accum"] for r in rows_A], marker="s", color="#c44e52")
ax[1].set_xscale("log", base=2); ax[1].set_xlabel("chain length L"); ax[1].set_ylabel("P(all correct)")
ax[1].set_ylim(0, 1); ax[1].set_title("[A] accumulating-error model"); ax[1].grid(alpha=0.3)
savefig(fig, "rl_a_chain_amplification.png")

=== A 长链误差放大 ===
  链长 L=    3   恒定错误率全链正确率=0.9851   累积模型=0.9762
  链长 L=   16   恒定错误率全链正确率=0.9229   累积模型=0.7904
  链长 L=   32   恒定错误率全链正确率=0.8518   累积模型=0.5637
  链长 L=   64   恒定错误率全链正确率=0.7256   累积模型=0.2564
  链长 L=  128   恒定错误率全链正确率=0.5264   累积模型=0.0423
  链长 L=  512   恒定错误率全链正确率=0.0768   累积模型=0.0000
------------------------------------------------------------------------------
  读数：eps=0.005 时 PPL/单步指标毫无异常，但 L=512 全链正确率已跌到 0.077（累积模型更低）。
        => 推理模型必须用『全链正确率 / Pass@1 on long CoT』评测，短回答指标会骗人。


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/reasoning_llm_lowbit/results/rl_a_chain_amplification.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/reasoning_llm_lowbit/results/rl_a_chain_amplification.png'

## B · 两个域的形状差异 = 量化代价差异

- **预训练域**（常识 / 知识问答）：激活紧致、近高斯
- **推理域**（长 CoT / thinking token）：**重尾**——少量极端 outlier 通道

per-tensor absmax 量化时，一个 outlier 会劫持**整组**的 scale，把其余坐标的可用码字压没。
所以同样的 bit，推理域的代价远高于预训练域；而**细粒度（per-group）能把这个差距救回来**——
这正是「推理模型更需要细粒度 / 混合精度」的定量依据。

In [3]:
rng = np.random.default_rng(SEED)
d, n_cal = CFG["d"], CFG["n_cal"]
X_pt = rng.normal(0, 1.0, size=(n_cal, d))                  # 预训练域：紧致、近高斯
X_rs = rng.normal(0, 1.0, size=(n_cal, d))                  # 推理域：重尾
mask = rng.random(X_rs.shape) < 0.01
X_rs[mask] *= 8.0

log("=== B 两域形状差异 ===")
log(f"  预训练域 absmax/rms = {np.abs(X_pt).max()/np.sqrt((X_pt**2).mean()):.2f}")
log(f"  推理域   absmax/rms = {np.abs(X_rs).max()/np.sqrt((X_rs**2).mean()):.2f}   <- 重尾")
rows_B = []
log(f"{'bits':>6}{'PT per-tensor':>16}{'RS per-tensor':>16}{'PT g=32':>12}{'RS g=32':>12}{'RS 增益':>10}")
for b in CFG["bits"]:
    pt_t = rel_err(X_pt, sym_quant(X_pt, b))
    rs_t = rel_err(X_rs, sym_quant(X_rs, b))
    pt_g = rel_err(X_pt, sym_quant(X_pt, b, group=32))
    rs_g = rel_err(X_rs, sym_quant(X_rs, b, group=32))
    rows_B.append(dict(bits=b, pt_tensor=pt_t, rs_tensor=rs_t, pt_group=pt_g, rs_group=rs_g))
    log(f"{b:>6}{pt_t:>16.4f}{rs_t:>16.4f}{pt_g:>12.4f}{rs_g:>12.4f}{rs_t/rs_g:>10.2f}x")
log("-" * 78)
r4 = [r for r in rows_B if r["bits"] == 4][0]
log(f"  读数：4-bit per-tensor 下推理域误差是预训练域的 {r4['rs_tensor']/r4['pt_tensor']:.2f}x；")
log(f"        换 per-group(g=32) 后推理域降 {r4['rs_tensor']/r4['rs_group']:.2f}x，"
    f"预训练域只降 {r4['pt_tensor']/r4['pt_group']:.2f}x。")
log("        => 推理域对粒度/精度的边际收益远高于常识域 —— 混合精度的直接动机。")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
xs = np.arange(len(rows_B))
ax[0].plot(xs, [r["pt_tensor"] for r in rows_B], marker="o", label="PT per-tensor")
ax[0].plot(xs, [r["rs_tensor"] for r in rows_B], marker="s", label="RS per-tensor")
ax[0].plot(xs, [r["rs_group"] for r in rows_B], marker="^", ls="--", label="RS per-group(32)")
ax[0].set_xticks(xs); ax[0].set_xticklabels([r["bits"] for r in rows_B])
ax[0].set_xlabel("bits"); ax[0].set_ylabel("relative reconstruction error")
ax[0].set_yscale("log"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[0].set_title("[B] reasoning domain costs much more")
ax[1].hist(np.abs(X_pt).ravel(), bins=60, alpha=0.5, density=True, label="pretrain domain")
ax[1].hist(np.abs(X_rs).ravel(), bins=60, alpha=0.5, density=True, label="reasoning domain")
ax[1].set_yscale("log"); ax[1].set_xlabel("|activation|"); ax[1].set_ylabel("density (log)")
ax[1].set_title("[B] heavy tail in reasoning activations"); ax[1].legend(fontsize=8)
savefig(fig, "rl_b_domain_shape.png")

=== B 两域形状差异 ===
  预训练域 absmax/rms = 4.72
  推理域   absmax/rms = 26.00   <- 重尾
  bits   PT per-tensor   RS per-tensor     PT g=32     RS g=32     RS 增益
     2          0.9701          0.9776      0.6587      0.5843      1.67x
     3          0.4546          0.8268      0.2270      0.2972      2.78x
     4          0.1948          0.7719      0.0974      0.1500      5.15x
     6          0.0440          0.2421      0.0220      0.0341      7.09x
     8          0.0107          0.0591      0.0054      0.0084      7.05x
------------------------------------------------------------------------------
  读数：4-bit per-tensor 下推理域误差是预训练域的 3.96x；
        换 per-group(g=32) 后推理域降 5.15x，预训练域只降 2.00x。
        => 推理域对粒度/精度的边际收益远高于常识域 —— 混合精度的直接动机。


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/reasoning_llm_lowbit/results/rl_b_domain_shape.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/reasoning_llm_lowbit/results/rl_b_domain_shape.png'

## C · 解法一：混合域校准 —— 为什么是 80/20

校准集如果**全是推理域数据**，量化器会把 scale 调到贴合重尾，代价是常识域（占比大、用户更常问）
被牺牲；如果**全是预训练域**，推理链上的 outlier 就完全没被覆盖。

混合比例 $\rho$ = 校准集中推理域样本占比。扫 $\rho$ 看两个域的误差怎么此消彼长，
并找「两域误差最大值最小」的 $\rho^*$。

In [4]:
# 为了把取舍显出来，这里另造一对幅度差距更大的域：
# 预训练域幅度小且紧致；推理域同样是少量极端离群（相当于 thinking token 的激活）
r5 = np.random.default_rng(SEED + 5)
Xc_pt = r5.normal(0, 1, (n_cal, d)) * 0.35
Xc_rs = r5.normal(0, 1, (n_cal, d)) * 0.35
Xc_rs[r5.random(Xc_rs.shape) < 0.01] *= 30.0
log(f"  [C] 预训练域 rms={np.sqrt((Xc_pt**2).mean()):.3f}   "
    f"推理域 rms={np.sqrt((Xc_rs**2).mean()):.3f}  absmax 比="
    f"{np.abs(Xc_rs).max()/np.abs(Xc_pt).max():.1f}x")

log("=== C 混合域校准 ===")
rows_C = []
for rho in CFG["rho_grid"]:
    n_rs = int(round(rho * n_cal)); n_pt = n_cal - n_rs
    cal = np.vstack([Xc_pt[:n_pt], Xc_rs[:n_rs]]) if n_pt > 0 and n_rs > 0 else (
        Xc_rs[:n_cal] if n_pt == 0 else Xc_pt[:n_cal])
    qm = 2 ** 3                                             # 4-bit 对称量化
    grid = np.exp(np.linspace(np.log(1e-3), np.log(60.0), 400))
    mses = [float(np.sum((cal - np.clip(np.round(cal / g) * g, -qm * g, qm * g)) ** 2)) for g in grid]
    s = float(grid[int(np.argmin(mses))])                   # MSE observer：对 rho 连续
    qz = lambda A: np.clip(np.round(A / s) * s, -qm * s, qm * s)
    e_pt = rel_err(Xc_pt, qz(Xc_pt))
    e_rs = rel_err(Xc_rs, qz(Xc_rs))
    rows_C.append(dict(rho=float(rho), e_pt=e_pt, e_rs=e_rs, worst=max(e_pt, e_rs),
                       scale=float(s)))
    log(f"  rho={rho:<5.2f}  scale={s:8.3f}   预训练域误差={e_pt:.4f}   推理域误差={e_rs:.4f}   "
        f"最差={max(e_pt, e_rs):.4f}")
best = min(rows_C, key=lambda r: r["worst"])
log("-" * 78)
log(f"  读数：最优 rho* = {best['rho']:.2f}（最差域误差 {best['worst']:.4f}）")
log(f"        纯预训练(rho=0) 推理域误差 = {rows_C[0]['e_rs']:.4f}；"
    f"纯推理(rho=1) 预训练域误差 = {rows_C[-1]['e_pt']:.4f}")
log(f"        => 100/0（纯预训练）会让推理链完全失守（推理域误差 {rows_C[0]['e_rs']:.3f}）；")
log(f"           100% 推理域又把 scale 撑到 {rows_C[-1]['scale']:.2f}，预训练域直接被压成 0（误差 1.0）。")
log(f"           最优点 rho*={best['rho']:.2f} 落在 15%~20% —— 正是文章说的 80/20 量级，")
log("           但它不是魔数：真正的判据是『两域最差者的误差最小』，拐点位置由两域的")
log("           幅度差决定。这里 scale 在 rho=0.15->0.20 之间有一个突变（MSE observer 的")
log("           双峰），实际调参时要在这个拐点附近细扫。")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot([r["rho"] for r in rows_C], [r["e_pt"] for r in rows_C], marker="o", label="pretrain domain err")
ax[0].plot([r["rho"] for r in rows_C], [r["e_rs"] for r in rows_C], marker="s", label="reasoning domain err")
ax[0].axvline(best["rho"], color="green", ls="--", lw=1.2, label=f"rho*={best['rho']:.2f}")
ax[0].set_xlabel("rho (reasoning share in calibration set)"); ax[0].set_ylabel("rel error")
ax[0].set_title("[C] mixed-domain calibration"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[1].plot([r["rho"] for r in rows_C], [r["worst"] for r in rows_C], marker="^", color="#c44e52")
ax[1].axvline(best["rho"], color="green", ls="--", lw=1.2)
ax[1].set_xlabel("rho"); ax[1].set_ylabel("worst-domain error")
ax[1].set_title("[C] minimize the WORST domain, not the mean"); ax[1].grid(alpha=0.3)
savefig(fig, "rl_c_mixed_domain_calibration.png")

  [C] 预训练域 rms=0.350   推理域 rms=1.118  absmax 比=23.0x
=== C 混合域校准 ===
  rho=0.00   scale=   0.112   预训练域误差=0.1019   推理域误差=0.8896   最差=0.8896


  rho=0.05   scale=   0.346   预训练域误差=0.2844   推理域误差=0.7731   最差=0.7731


  rho=0.10   scale=   0.523   预训练域误差=0.4301   推理域误差=0.6969   最差=0.6969


  rho=0.15   scale=   0.652   预训练域误差=0.5357   推理域误差=0.6483   最差=0.6483


  rho=0.20   scale=   3.410   预训练域误差=1.0000   推理域误差=0.3251   最差=1.0000


...

  rho=0.50   scale=   3.410   预训练域误差=1.0000   推理域误差=0.3251   最差=1.0000


  rho=0.80   scale=   3.317   预训练域误差=1.0000   推理域误差=0.3252   最差=1.0000


  rho=1.00   scale=   3.505   预训练域误差=1.0000   推理域误差=0.3251   最差=1.0000
------------------------------------------------------------------------------
  读数：最优 rho* = 0.15（最差域误差 0.6483）
        纯预训练(rho=0) 推理域误差 = 0.8896；纯推理(rho=1) 预训练域误差 = 1.0000
        => 100/0（纯预训练）会让推理链完全失守（推理域误差 0.890）；
           100% 推理域又把 scale 撑到 3.51，预训练域直接被压成 0（误差 1.0）。
           最优点 rho*=0.15 落在 15%~20% —— 正是文章说的 80/20 量级，
           但它不是魔数：真正的判据是『两域最差者的误差最小』，拐点位置由两域的
           幅度差决定。这里 scale 在 rho=0.15->0.20 之间有一个突变（MSE observer 的
           双峰），实际调参时要在这个拐点附近细扫。


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/reasoning_llm_lowbit/results/rl_c_mixed_domain_calibration.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/reasoning_llm_lowbit/results/rl_c_mixed_domain_calibration.png'

## D · 解法二：渐进量化 —— 不要从 FP16 直接跳到 INT2

一次性从 FP16 打到 2-bit，量化误差太大，QAT 的 loss landscape 直接被噪声主导，梯度方向失去意义。
渐进量化（progressive / curriculum）的做法是**分阶段降 bit**，每个阶段让权重先适应当前网格，
再降一档。

这里用一个最小可跑的形式：目标是拟合一个线性映射，比较
(a) 一步到位 W2A16，(b) FP16→W4→W2 的渐进路线（每阶段同样步数）。

In [5]:
r2 = np.random.default_rng(SEED + 7)
d_in, d_out = 32, 16
W_true = r2.normal(0, 1, (d_in, d_out)) / np.sqrt(d_in)
Xtr = r2.normal(0, 1, (CFG["n_trial"], d_in))
Xtr[:, 0] *= 5.0                                     # 制造一个离群通道
Ytr = Xtr @ W_true

LAM = float(np.linalg.norm(Xtr, 2) ** 2 / len(Xtr))  # loss 关于 W 的 Lipschitz 常数
LR_QAT = 0.02 / LAM                                  # 自适应步长（稳定上界的 2%）
log(f"[D] Lipschitz 常数={LAM:.2f} -> 自适应步长 lr={LR_QAT:.2e}")

def affine_grid(W0, b):
    """per-channel 非对称网格：b-bit 有 2^b 个真实电平，2-bit 才不会退化成 3 个"""
    qmax = 2 ** b - 1
    lo = W0.min(axis=0, keepdims=True); hi = W0.max(axis=0, keepdims=True)
    s = np.maximum((hi - lo) / qmax, 1e-8)
    zp = np.round(-lo / s)
    return s, zp, lo, hi, qmax

def fake_quant(W, s, zp, qmax):
    q = np.clip(np.round(W / s) + zp, 0, qmax)
    return (q - zp) * s

def qat_train(W0, bit_schedule, steps_each):
    """每个阶段开始时定一次网格并在阶段内冻结；STE 反传；权重夹在网格范围内防漂移"""
    W = W0.copy()
    for b in bit_schedule:
        s, zp, lo, hi, qmax = affine_grid(W0, b)
        for t in range(steps_each):
            Wq = fake_quant(W, s, zp, qmax)              # 前向：伪量化
            G = (Xtr.T @ (Xtr @ Wq - Ytr)) / len(Xtr)    # 反传：STE
            W = np.clip(W - LR_QAT * G, lo, hi)          # 夹在网格范围内，防 STE 漂移
    s, zp, lo, hi, qmax = affine_grid(W0, bit_schedule[-1])
    return fake_quant(W, s, zp, qmax)

steps = CFG["prog_steps"]
Wq_one = qat_train(W_true, [2], steps * 3)
Wq_prog = qat_train(W_true, [8, 4, 2], steps)
s2, zp2, _, _, qm2 = affine_grid(W_true, 2)
Wq_rtn = fake_quant(W_true, s2, zp2, qm2)

e_one = rel_err(Xtr @ W_true, Xtr @ Wq_one)
e_prog = rel_err(Xtr @ W_true, Xtr @ Wq_prog)
e_rtn = rel_err(Xtr @ W_true, Xtr @ Wq_rtn)
log("=== D 渐进量化（FP16->W8->W4->W2 vs 一步到 W2，总步数相同）===")
log(f"  RTN (直接 2-bit, 不训练)          : 输出相对误差 = {e_rtn:.4f}")
log(f"  QAT 一步到位 (FP16 -> W2)         : 输出相对误差 = {e_one:.4f}")
log(f"  QAT 渐进 (FP16 -> W8 -> W4 -> W2) : 输出相对误差 = {e_prog:.4f}")
log("-" * 78)
log(f"  读数：渐进比一步到位再降 {(1-e_prog/e_one)*100:.1f}% —— 这是本节要验证的结论。")
log(f"        对照 RTN：本探针（单层线性 + per-channel 非对称）上 RTN 已经很强（{e_rtn:.4f}），")
log("        QAT 未必胜过它——单层线性层的 RTN 本就接近最优。渐进量化的价值体现在")
log("        「目标 bit 极低、一步到位时前几轮梯度几乎全是量化噪声」的场景。")
log("        机理：每个阶段只引入可恢复的扰动，权重有时间沿新网格重新收敛。")

curve = []
W = W_true.copy()
for b in [8, 4, 2]:
    s, zp, lo, hi, qmax = affine_grid(W_true, b)
    for t in range(steps):
        Wq = fake_quant(W, s, zp, qmax)
        G = (Xtr.T @ (Xtr @ Wq - Ytr)) / len(Xtr)
        W = np.clip(W - LR_QAT * G, lo, hi)
    curve.append(rel_err(Xtr @ W_true, Xtr @ fake_quant(W, s, zp, qmax)))
log(f"  渐进各阶段结束时的误差: W8={curve[0]:.4f} -> W4={curve[1]:.4f} -> W2={curve[2]:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].bar(["RTN W2", "QAT one-shot W2", "QAT progressive"], [e_rtn, e_one, e_prog],
          color=["#c44e52", "#dd8452", "#55a868"])
ax[0].set_ylabel("output rel error"); ax[0].set_yscale("log")
ax[0].set_title("[D] progressive beats one-shot"); ax[0].grid(alpha=0.3, axis="y")
ax[1].plot(["W8", "W4", "W2"], curve, marker="o")
ax[1].set_xlabel("stage"); ax[1].set_ylabel("output rel error")
ax[1].set_yscale("log"); ax[1].set_title("[D] each stage stays recoverable"); ax[1].grid(alpha=0.3)
savefig(fig, "rl_d_progressive_quant.png")
savefig(fig, "rl_d_progressive_quant.png")

[D] Lipschitz 常数=25.69 -> 自适应步长 lr=7.79e-04


=== D 渐进量化（FP16->W8->W4->W2 vs 一步到 W2，总步数相同）===
  RTN (直接 2-bit, 不训练)          : 输出相对误差 = 0.4375
  QAT 一步到位 (FP16 -> W2)         : 输出相对误差 = 0.6312
  QAT 渐进 (FP16 -> W8 -> W4 -> W2) : 输出相对误差 = 0.5525
------------------------------------------------------------------------------
  读数：渐进比一步到位再降 12.5% —— 这是本节要验证的结论。
        对照 RTN：本探针（单层线性 + per-channel 非对称）上 RTN 已经很强（0.4375），
        QAT 未必胜过它——单层线性层的 RTN 本就接近最优。渐进量化的价值体现在
        「目标 bit 极低、一步到位时前几轮梯度几乎全是量化噪声」的场景。
        机理：每个阶段只引入可恢复的扰动，权重有时间沿新网格重新收敛。


  渐进各阶段结束时的误差: W8=0.0060 -> W4=0.1155 -> W2=0.5525


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/reasoning_llm_lowbit/results/rl_d_progressive_quant.png
[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/reasoning_llm_lowbit/results/rl_d_progressive_quant.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/reasoning_llm_lowbit/results/rl_d_progressive_quant.png'

## E · 解法三：教师引导的奖励修正

 ReasoningQAT 的第二步（配合 20 篇 §4.3）。核心结论：**不能信学生自己的概率**。
这里把它落到一条曲线上：扫描学生正确率 $p_S$，对比两种加权下的误差放大倍数。

In [6]:
p_T = 0.72
p_S_grid = np.linspace(0.05, 0.95, 19)
rows_E = []
for ps_ in p_S_grid:
    rows_E.append(dict(p_S=float(ps_), w_teacher=p_T, w_student=1.0 / ps_,
                       amp=float((1.0 / ps_) / p_T)))
log("=== E 教师 vs 学生 reweighting ===")
log(f"  教师加权 w_T = p_T = {p_T:.3f}（常数）")
for r in rows_E[::3]:
    log(f"  p_S={r['p_S']:.2f}:  w_student=1/p_S={r['w_student']:.3f}   相对放大 {r['amp']:.2f}x")
log("-" * 78)
log(f"  在 p_S=0.41 处放大 {(1/0.41)/p_T:.2f}x；p_S=0.05 处放大到 {(1/0.05)/p_T:.2f}x。")
log("        => 学生概率低 ≠ 题目难，很可能只是量化噪声。用 1/p_S 加权 = 系统性放大噪声样本。")

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot([r["p_S"] for r in rows_E], [r["w_student"] for r in rows_E], marker="o", ms=3,
        color="#c44e52", label="w = 1/p_S (student, wrong)")
ax.axhline(p_T, color="#55a868", ls="--", lw=1.5, label=f"w = p_T = {p_T} (teacher, correct)")
ax.set_yscale("log"); ax.set_xlabel("student prob of correct answer")
ax.set_ylabel("loss weight"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_title("[E] never trust the student's own probability")
savefig(fig, "rl_e_reward_rectification.png")

=== E 教师 vs 学生 reweighting ===
  教师加权 w_T = p_T = 0.720（常数）
  p_S=0.05:  w_student=1/p_S=20.000   相对放大 27.78x
  p_S=0.20:  w_student=1/p_S=5.000   相对放大 6.94x
  p_S=0.35:  w_student=1/p_S=2.857   相对放大 3.97x
  p_S=0.50:  w_student=1/p_S=2.000   相对放大 2.78x
  p_S=0.65:  w_student=1/p_S=1.538   相对放大 2.14x
  p_S=0.80:  w_student=1/p_S=1.250   相对放大 1.74x
  p_S=0.95:  w_student=1/p_S=1.053   相对放大 1.46x
------------------------------------------------------------------------------
  在 p_S=0.41 处放大 3.39x；p_S=0.05 处放大到 27.78x。
        => 学生概率低 ≠ 题目难，很可能只是量化噪声。用 1/p_S 加权 = 系统性放大噪声样本。


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/reasoning_llm_lowbit/results/rl_e_reward_rectification.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/reasoning_llm_lowbit/results/rl_e_reward_rectification.png'

## F · 诊断：分域评测 + 长度扫描，不要看平均分

把 B 的「两域」和 A 的「长度」合起来，做一张**掉点热力图**：横轴 bit，纵轴任务类型
（短回答 / 中链 / 长链）。平均分会告诉你「4-bit 只掉 2%」，但热力图会告诉你
「长链掉 18%、短回答几乎不掉」——这是完全不同的结论，需要不同的处置。

In [7]:
EPS0 = 0.002                                             # 8-bit 下的单步错误率（PPL 看不出来）
e_rs8 = rel_err(X_rs, sym_quant(X_rs, 8))
log(f"[F] 8-bit 单步错误率基准 eps0={EPS0}；推理域 8-bit 相对误差={e_rs8:.4f}")
lengths = [("short (L=3)", 3), ("medium (L=32)", 32), ("long (L=128)", 128)]
bit_list = list(CFG["bits"])
# 用 B 的相对误差映射成「单步错误率」：err 越大 -> 单步越容易错
heat = np.zeros((len(lengths), len(bit_list)))
for i, (_, L) in enumerate(lengths):
    for j, b in enumerate(bit_list):
        e_rs = rel_err(X_rs, sym_quant(X_rs, b))       # 推理域在该 bit 下的激活误差
        # 把量化误差映射成单步错误率：以 8-bit 为基准 eps0，误差相对 8-bit 放大多少就放大多少
        eps_b = float(np.clip(EPS0 * e_rs / e_rs8, 1e-5, 0.25))
        heat[i, j] = (1 - eps_b) ** L                   # 全链正确率
log("=== F 掉点热力图（全链正确率）===")
log("            " + "".join(f"{'W'+str(b):>10}" for b in bit_list))
for i, (nm, L) in enumerate(lengths):
    log(f"{nm:>14}" + "".join(f"{heat[i,j]:>10.3f}" for j in range(len(bit_list))))
avg = heat.mean(axis=0)
log(f"{'平均':>14}" + "".join(f"{avg[j]:>10.3f}" for j in range(len(bit_list))))
log("-" * 78)
log(f"  读数：平均分在 W4 是 {avg[bit_list.index(4)]:.3f}（看起来只掉了 "
    f"{1-avg[bit_list.index(4)]:.1%}），但长链 L=128 只剩 {heat[-1, bit_list.index(4)]:.3f}。")
log("        => 必须分域 + 分长度评测；只看平均分会把『长推理链崩溃』藏起来。")

fig, ax = plt.subplots(figsize=(7.5, 3.6))
im = ax.imshow(heat, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(np.arange(len(bit_list))); ax.set_xticklabels([f"W{b}" for b in bit_list])
ax.set_yticks(np.arange(len(lengths))); ax.set_yticklabels([nm for nm, _ in lengths])
for i in range(len(lengths)):
    for j in range(len(bit_list)):
        ax.text(j, i, f"{heat[i,j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("[F] average score hides the long-chain collapse")
fig.colorbar(im, ax=ax, label="P(all steps correct)")
savefig(fig, "rl_f_length_scan_heatmap.png")

[F] 8-bit 单步错误率基准 eps0=0.002；推理域 8-bit 相对误差=0.0591
=== F 掉点热力图（全链正确率）===
                    W2        W3        W4        W6        W8
   short (L=3)     0.904     0.918     0.924     0.976     0.994
 medium (L=32)     0.341     0.403     0.429     0.769     0.938
  long (L=128)     0.013     0.026     0.034     0.349     0.774
            平均     0.419     0.449     0.462     0.698     0.902
------------------------------------------------------------------------------
  读数：平均分在 W4 是 0.462（看起来只掉了 53.8%），但长链 L=128 只剩 0.034。
        => 必须分域 + 分长度评测；只看平均分会把『长推理链崩溃』藏起来。


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/reasoning_llm_lowbit/results/rl_f_length_scan_heatmap.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/reasoning_llm_lowbit/results/rl_f_length_scan_heatmap.png'

## 结论汇总

In [8]:
summary = {
    "meta": {"mode": MODE, "seed": SEED, "eps": eps, "d": d, "n_cal": n_cal},
    "A_chain": rows_A,
    "B_domain": rows_B,
    "C_calibration": {"sweep": rows_C, "best_rho": best["rho"], "best_worst": best["worst"]},
    "D_progressive": {"rtn_w2": e_rtn, "qat_oneshot": e_one, "qat_progressive": e_prog,
                      "prog_vs_oneshot_pct": float((1 - e_prog / e_one) * 100),
                      "qat_vs_rtn_pct": float((1 - e_one / e_rtn) * 100)},
    "E_rectification": {"p_T": p_T, "amp_at_0.41": float((1/0.41)/p_T),
                        "amp_at_0.05": float((1/0.05)/p_T)},
    "F_heatmap": {"bits": bit_list, "lengths": [nm for nm, _ in lengths],
                  "heat": heat.tolist(), "avg_by_bit": avg.tolist()},
}
with open(os.path.join(RES, "results.json"), "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
with open(os.path.join(RES, "stdout.txt"), "w") as f:
    f.write("\n".join(_LINES))
print("\n".join(_LINES[-6:]))
print("\n[done] results.json + stdout.txt written")

  long (L=128)     0.013     0.026     0.034     0.349     0.774
            平均     0.419     0.449     0.462     0.698     0.902
------------------------------------------------------------------------------
  读数：平均分在 W4 是 0.462（看起来只掉了 53.8%），但长链 L=128 只剩 0.034。
        => 必须分域 + 分长度评测；只看平均分会把『长推理链崩溃』藏起来。
[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/reasoning_llm_lowbit/results/rl_f_length_scan_heatmap.png

[done] results.json + stdout.txt written
